<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/front/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CKIP-BERT Multi-Task Inference

Loads the five best CKIP-BERT fold checkpoints produced by `model_train.ipynb`, averages task probabilities, applies OOF-tuned T1/T3 routing thresholds, and exports the same CSV interface as the SetFit pipeline.


In [4]:
# Install dependencies in Colab, then restart the runtime if requested.
# !pip install -q transformers torch pandas numpy tqdm huggingface_hub


In [5]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download, snapshot_download
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


DEFAULT_MODEL_NAME = "ckiplab/bert-base-chinese"
DEFAULT_FOLDS = [1, 2, 3, 4, 5]
DEFAULT_MAX_LEN = 512
DEFAULT_HEAD_RATIO = 0.25
DEFAULT_T1_THRESHOLD = 0.5
DEFAULT_T3_THRESHOLD = 0.5
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

ID_COLUMN = "id"
TEXT_COLUMN = "data"
TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]
TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}


def tokenize_head_tail(text, tokenizer, max_len, head_ratio):
    body_ids = tokenizer.encode(
        f"文本：{str(text)}",
        add_special_tokens=False,
        verbose=False,
    )
    max_body_len = max_len - 2
    if len(body_ids) > max_body_len:
        head_len = int(max_body_len * head_ratio)
        tail_len = max_body_len - head_len
        body_ids = body_ids[:head_len] + body_ids[-tail_len:]
    input_ids = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]
    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return input_ids, attention_mask


class ESGInferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, head_ratio):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.head_ratio = head_ratio

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        input_ids, attention_mask = tokenize_head_tail(
            self.df.iloc[index][TEXT_COLUMN],
            self.tokenizer,
            self.max_len,
            self.head_ratio,
        )
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }


class ESGUnifiedMTLModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.multi_sample_dropouts = nn.ModuleList(
            [nn.Dropout(probability) for probability in [0.1, 0.2, 0.3, 0.4, 0.5]]
        )
        self.t1_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t3_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t2_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 4),
        )
        self.t4_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 3),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        t1_logits = torch.stack(
            [self.t1_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        t3_logits = torch.stack(
            [self.t3_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)
        return t1_logits, t2_logits, t3_logits, t4_logits


def sigmoid_numpy(values):
    values = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-values))


def softmax_numpy(values):
    shifted = values - values.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def find_artifact_root(path):
    path = Path(path)
    candidates = [path, path / "mtl_outputs"]
    for candidate in candidates:
        if (candidate / "mtl_inference_config.json").exists():
            return candidate
    return None


def resolve_artifact_root(repo_id=None, model_dir=None):
    if repo_id:
        downloaded = snapshot_download(
            repo_id=repo_id,
            allow_patterns=[
                "mtl_inference_config.json",
                "mtl_thresholds.json",
                "tokenizer/**",
                "fold_*/best_model.pth",
                "mtl_outputs/mtl_inference_config.json",
                "mtl_outputs/mtl_thresholds.json",
                "mtl_outputs/tokenizer/**",
                "mtl_outputs/fold_*/best_model.pth",
                # Legacy layout remains readable.
                "best_mtl_model_fold_*.pth",
            ],
        )
        root = find_artifact_root(downloaded)
        return root if root is not None else Path(downloaded)

    if model_dir is None:
        model_dir = Path("mtl_outputs")
    root = find_artifact_root(model_dir)
    if root is None:
        raise FileNotFoundError(f"Could not find CKIP MTL artifacts under {model_dir}.")
    return root


def load_config(root):
    config_path = Path(root) / "mtl_inference_config.json"
    if not config_path.exists():
        return {
            "artifact_version": 1,
            "model_name": DEFAULT_MODEL_NAME,
            "folds": DEFAULT_FOLDS,
            "max_len": DEFAULT_MAX_LEN,
            "head_ratio": DEFAULT_HEAD_RATIO,
            "thresholds": {
                "t1_threshold": DEFAULT_T1_THRESHOLD,
                "t3_threshold": DEFAULT_T3_THRESHOLD,
            },
        }
    with open(config_path, "r", encoding="utf-8") as file:
        config = json.load(file)
    if int(config.get("artifact_version", 1)) not in {1, 2}:
        raise ValueError(f"Unsupported artifact version: {config.get('artifact_version')}")
    return config


def resolve_checkpoint(root, fold, repo_id=None):
    root = Path(root)
    new_path = root / f"fold_{fold}" / "best_model.pth"
    if new_path.exists():
        return new_path
    legacy_path = root / f"best_mtl_model_fold_{fold}.pth"
    if legacy_path.exists():
        return legacy_path
    if repo_id:
        return Path(hf_hub_download(repo_id=repo_id, filename=f"best_mtl_model_fold_{fold}.pth"))
    raise FileNotFoundError(f"Missing checkpoint for fold {fold}.")


def predict_fold(model, data_loader):
    output = {task: [] for task in ["t1", "t2", "t3", "t4"]}
    model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Inference"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids, attention_mask)
            output["t1"].append(sigmoid_numpy(logits[0].float().cpu().numpy()))
            output["t2"].append(softmax_numpy(logits[1].float().cpu().numpy()))
            output["t3"].append(sigmoid_numpy(logits[2].float().cpu().numpy()))
            output["t4"].append(softmax_numpy(logits[3].float().cpu().numpy()))
    return {task: np.concatenate(parts, axis=0) for task, parts in output.items()}


def argmax_label(probabilities, labels):
    return labels[int(np.argmax(probabilities))]


def route_predictions(test_df, probabilities, t1_threshold, t3_threshold):
    results = []
    for index in range(len(test_df)):
        if probabilities["t1"][index] < t1_threshold:
            results.append(
                {
                    ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t2_prediction = argmax_label(probabilities["t2"][index], TASK_CLASSES["t2"])
        if probabilities["t3"][index] < t3_threshold:
            results.append(
                {
                    ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2_prediction,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue

        results.append(
            {
                ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2_prediction,
                "evidence_status": "Yes",
                "evidence_quality": argmax_label(
                    probabilities["t4"][index],
                    TASK_CLASSES["t4"],
                ),
            }
        )
    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def validate_input(df):
    missing = [column for column in [ID_COLUMN, TEXT_COLUMN] if column not in df.columns]
    if missing:
        raise ValueError(f"Test CSV missing columns: {missing}")
    if df[ID_COLUMN].duplicated().any():
        raise ValueError("Test CSV contains duplicated ids.")


def validate_output(output_df, test_df):
    expected_columns = [ID_COLUMN] + TARGET_COLUMNS
    if list(output_df.columns) != expected_columns:
        raise ValueError(f"Unexpected output columns: {list(output_df.columns)}")
    if len(output_df) != len(test_df):
        raise ValueError("Output row count differs from test input.")
    if output_df[ID_COLUMN].tolist() != test_df[ID_COLUMN].tolist():
        raise ValueError("Output id order differs from test input.")


def ensemble_inference_and_export(
    repo_id,
    test_csv_path,
    output_csv_path="final_submission.csv",
    model_dir=None,
):
    root = resolve_artifact_root(repo_id=repo_id, model_dir=model_dir)
    config = load_config(root)
    model_name = config.get("model_name", DEFAULT_MODEL_NAME)
    folds = [int(fold) for fold in config.get("folds", DEFAULT_FOLDS)]
    max_len = int(config.get("max_len", DEFAULT_MAX_LEN))
    head_ratio = float(config.get("head_ratio", DEFAULT_HEAD_RATIO))
    thresholds = config.get("thresholds", {})
    t1_threshold = float(thresholds.get("t1_threshold", DEFAULT_T1_THRESHOLD))
    t3_threshold = float(thresholds.get("t3_threshold", DEFAULT_T3_THRESHOLD))

    test_df = pd.read_csv(test_csv_path).reset_index(drop=True)
    validate_input(test_df)

    tokenizer_path = Path(root) / "tokenizer"
    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_path) if tokenizer_path.exists() else model_name,
        use_fast=True,
    )
    dataset = ESGInferenceDataset(test_df, tokenizer, max_len, head_ratio)
    data_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=USE_AMP,
    )

    accumulated = {
        "t1": np.zeros(len(test_df), dtype=np.float64),
        "t2": np.zeros((len(test_df), 4), dtype=np.float64),
        "t3": np.zeros(len(test_df), dtype=np.float64),
        "t4": np.zeros((len(test_df), 3), dtype=np.float64),
    }

    for fold in folds:
        print(f"Loading CKIP-BERT fold {fold}")
        checkpoint_path = resolve_checkpoint(root, fold, repo_id=repo_id)
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        state_dict = checkpoint.get("model_state_dict", checkpoint)
        model = ESGUnifiedMTLModel(model_name).to(DEVICE)
        model.backbone.resize_token_embeddings(len(tokenizer))
        model.load_state_dict(state_dict)
        fold_probabilities = predict_fold(model, data_loader)
        for task in accumulated:
            accumulated[task] += fold_probabilities[task]
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    averaged = {
        task: values / len(folds)
        for task, values in accumulated.items()
    }
    output_df = route_predictions(
        test_df,
        averaged,
        t1_threshold=t1_threshold,
        t3_threshold=t3_threshold,
    )
    validate_output(output_df, test_df)
    output_df.to_csv(output_csv_path, index=False)
    print(f"Exported CKIP-BERT submission to {output_csv_path}")
    print(output_df.head())
    return output_df


In [6]:
# ==========================================
# Run inference
# ==========================================

# Local artifacts:
# ensemble_inference_and_export(
#     repo_id=None,
#     test_csv_path="/content/test.csv",
#     output_csv_path="final_submission.csv",
#     model_dir="/content/mtl_outputs",
# )

# Hugging Face artifacts:
DEFAULT_REPO_ID = "maxbeettww/VeriPromise_ESG_2026_9906"
TEST_CSV_PATH = "../data/ori_data/vpesg4k_test_2000.csv"

# Uncomment after uploading the new artifacts.
ensemble_inference_and_export(
    repo_id=DEFAULT_REPO_ID,
    test_csv_path=TEST_CSV_PATH,
    output_csv_path="final_submission.csv",
)


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading CKIP-BERT fold 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Inference:   0%|          | 0/125 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Inference:   0%|          | 0/125 [00:00<?, ?it/s]

Exported CKIP-BERT submission to final_submission.csv
      id promise_status  verification_timeline evidence_status  \
0  12001            Yes                already             Yes   
1  12002            Yes  between_2_and_5_years             Yes   
2  12003            Yes                already             Yes   
3  12004            Yes  between_2_and_5_years             Yes   
4  12005            Yes                already             Yes   

  evidence_quality  
0            Clear  
1            Clear  
2            Clear  
3            Clear  
4            Clear  


,id,promise_status,verification_timeline,evidence_status,evidence_quality
0,12001,Yes,already,Yes,Clear
1,12002,Yes,between_2_and_5_years,Yes,Clear
2,12003,Yes,already,Yes,Clear
3,12004,Yes,between_2_and_5_years,Yes,Clear
4,12005,Yes,already,Yes,Clear
...,...,...,...,...,...
1995,13996,Yes,between_2_and_5_years,Yes,Clear
1996,13997,Yes,between_2_and_5_years,Yes,Clear
1997,13998,Yes,already,Yes,Clear
1998,13999,Yes,already,Yes,Clear
